In [5]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

In [6]:
transactions = pd.read_sql("SELECT * FROM transactions", engine)
transactions.head()

,txn_id,agent_id,service_code,txn_ts,amount_xaf,fee_xaf,status,customer_msisdn
0,TX0012915,AG5591,SAL,2024-12-03 08:25:00,3294.0,19.0,SUCCESS,659464793
1,TX0205100,AG5802,INS,2025-01-31 17:57:00,416584.0,5832.0,FAILED,672642613
2,TX0070259,AG5232,BILL,2026-03-29 20:41:00,5979.0,53.0,SUCCESS,653348892
3,TX0194943,AG5592,TRAN,2025-02-02 12:58:00,8579.0,111.0,REVERSED,667846125
4,TX0128249,AG6103,TRAN,2025-12-24 11:07:00,49939.0,649.0,SUCCESS,659976632


In [7]:
# run the 3 groupbys using pandas
p1 = transactions.groupby("agent_id")["amount_xaf"].sum().reset_index()
p2 = transactions.groupby(["agent_id", "service_code"])["fee_xaf"].sum().reset_index()
p3 = transactions.groupby("status")["txn_id"].count().reset_index().rename(columns={"txn_id": "txn_count"})

In [4]:
# run the same 3 groupbys using SQL
s1 = pd.read_sql("""
    SELECT agent_id, SUM(amount_xaf) AS amount_xaf
    FROM transactions
    GROUP BY agent_id
    ORDER BY agent_id
""", engine)
 
s2 = pd.read_sql("""
    SELECT agent_id, service_code, SUM(fee_xaf) AS fee_xaf
    FROM transactions
    GROUP BY agent_id, service_code
    ORDER BY agent_id, service_code
""", engine)
 
s3 = pd.read_sql("""
    SELECT status, COUNT(*) AS txn_count
    FROM transactions
    GROUP BY status
    ORDER BY status
""", engine)

In [8]:
# function that compares a pandas result to a sql result and flags differences
def diff(pandas_result, sql_result, keys):
    merged = pandas_result.merge(sql_result, on=keys, suffixes=("_py", "_sql"))
    value_cols = [c for c in merged.columns if c.endswith("_py")]
    for col in value_cols:
        base = col[:-3]  # remove "_py"
        merged[base + "_diff"] = merged[col] - merged[base + "_sql"]
    return merged
 

In [10]:
# compare all 3 pairs, empty output means pandas and SQL agree

# compare 1: total amount per agent
print((p1["amount_xaf"].values == s1["amount_xaf"].values).all())

# compare 2: fee per agent per service
print((p2["fee_xaf"].values == s2["fee_xaf"].values).all())

# compare 3: count per status
print((p3["txn_count"].values == s3["txn_count"].values).all())

True
True
True
